# ColGraphRAG WebQA — Step-by-Step Pipeline Tutorial

This notebook follows the **query-driven multimodal GraphRAG** pipeline step by step.

| Phase | Script | Role |
|-------|--------|------|
| 0 | `export_webqa_slice.py` *or* prebuilt slice | Build/copy JSONL corpus slice |
| 2 | `pattern.py` | Per-question graph pattern (LLM) |
| 3 | `extraction.py` | Entity/relation extraction (LLM) |
| 4 | `construct.py` | NetworkX → GraphML |
| 5 | `inference.py` | ColEmbed MaxSim retrieval + Gemma answer |
| 6 | `eval/evaluate_webqa_qa.py` | QA-FL / QA-Acc / QA metrics |

**Prerequisites:** Finish the **environment** section below (venv + `pip install -r requirements.txt`). Before real LLM / ColEmbed work, fetch **Hugging Face weights** with `util/download_models.py` (see **Hugging Face model download**). Actual Gemma runs need CUDA; `--dry-run` in the pipeline cells checks wiring without loading heavy models.

**Working directory:** Run Jupyter from the repo root `colgraphrag_webqa`, or adjust `REPO` in the **Resolve repo root** cell.

**Two paths:** use **End-to-end** for one full pass through the notebook, or **step-by-step (manual)** so you execute each phase yourself. Do not mix both in the same notebook session unless you use different `RUN_ID` values or you understand how the outputs overlap.


## Virtual environment and CUDA dependencies

Do this **once** on your machine before opening this notebook. Commands assume you are in the repo root `colgraphrag_webqa/` (the folder that contains `requirements.txt`, `inference.py`, and `notebook/`).

### Create a venv

**Linux / macOS**

```bash
cd /path/to/colgraphrag_webqa
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip setuptools wheel
```

**Windows (PowerShell)**

```powershell
cd C:\path\to\colgraphrag_webqa
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -U pip setuptools wheel
```

Use **Python 3.10+** (3.11 recommended).

### Point this notebook at `.venv` (VS Code / Cursor)

After creating the venv, select it as the runtime for notebook cells:

1. Open this `.ipynb` file.
2. Click **Select Kernel** in the top-right of the notebook (or run **Notebook: Select Notebook Kernel** from the Command Palette).
3. Choose **Python Environments** (you may need **Select Another Kernel…** first).
4. Pick **Existing Python environment** and select the interpreter under the repo:  
   **`.venv/bin/python`** (Linux/macOS) or **`.venv\Scripts\python.exe`** (Windows).  
   If `.venv` does not appear, use **Enter interpreter path…** and browse to that path.

If you register an ipykernel display name later (**Jupyter kernel** below), you can select that kernel instead of browsing to `.venv`.

### Install CUDA PyTorch inside the venv (explicit)

With the venv **activated** and `pip` upgraded, install the CUDA 12.4 wheels first (same pins as `requirements.txt`):

```bash
pip install torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124 \
  --extra-index-url https://pypi.org/simple
```

Then install the rest of the project (networkx, transformers, etc.):

```bash
pip install -r requirements.txt
```

`pip` will keep the CUDA PyTorch you installed if versions match; otherwise prefer a single pass below.

### Install GPU stack from `requirements.txt` (single command)

The file pins **PyTorch with CUDA 12.4** via `--index-url https://download.pytorch.org/whl/cu124` at the top. Installing the whole file in one go is equivalent to the two-step flow above:

```bash
pip install -r requirements.txt
```

After install, verify CUDA is visible to PyTorch:

```bash
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

You should see a `+cu124` build and `True` when an NVIDIA driver + GPU are present.

### Jupyter kernel (optional)

To run **this** notebook inside the venv:

```bash
pip install ipykernel
python -m ipykernel install --user --name colgraphrag-webqa --display-name "Python (colgraphrag_webqa)"
```

Then select the **Python (colgraphrag-webqa)** kernel in VS Code / Cursor / Jupyter.

**CPU-only note:** `requirements.txt` targets CUDA. For a CPU-only smoke test you would need a separate pin (not covered here); pipeline sections below support `--dry-run` without loading Gemma on GPU.

---


## Available GPU

Run this after activating the venv and installing PyTorch (`requirements.txt`). Shows **NVIDIA driver/GPU** via `nvidia-smi` when available, then **PyTorch CUDA** status via `torch` (skips torch lines if not installed yet).

In [ ]:
import shutil
import subprocess

nv = shutil.which("nvidia-smi")
if nv:
    r = subprocess.run(
        [
            nv,
            "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    if r.returncode == 0:
        print("nvidia-smi (GPU list):\n")
        print(r.stdout.strip() or "(empty)")
    else:
        print("nvidia-smi failed:", r.stderr)
else:
    print("nvidia-smi not found in PATH — no NVIDIA CLI, or not a GPU machine.")

print()

try:
    import torch
except ImportError:
    print("torch: not installed yet — complete pip install -r requirements.txt first.")
else:
    print("torch:", torch.__version__)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("torch.cuda.device_count():", n)
        for i in range(n):
            print(f"  [{i}]", torch.cuda.get_device_name(i))
            print("      mem allocated MB:", torch.cuda.memory_allocated(i) / 1e6)

## Resolve repo root and paths

The pipeline expects `PYTHONPATH` to include the repo root so imports like `mllm`, `util`, `prompt` resolve.

In [ ]:
import os
import sys
from pathlib import Path

# Find repo root (folder containing inference.py) from cwd or any parent
_here = Path.cwd()
REPO = None
for _p in [_here, *list(_here.resolve().parents)]:
    if (_p / "inference.py").is_file():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError(
        "Could not find repo root (folder with inference.py). "
        "Set Jupyter cwd to the colgraphrag repo root or cd there, then re-run this cell."
    )

assert (REPO / "inference.py").is_file(), f"Set REPO to the repository root; current {REPO}"

os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("REPO =", REPO.resolve())
print("cwd =", Path.cwd())

## Environment check (optional)

For **real** inference, use `GEMMA4_E4B_IT_TORCH_DTYPE=bf16` on typical single-GPU setups to reduce VRAM.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Configuration files

- `config/data.yaml` — dataset paths (`WEBQA_DATA_ROOT`, slice locations).
- `config/model.yaml` — Gemma, ColEmbed, fluency model paths and HF repo IDs.
- Optionally copy `.env.example` to `.env` and set `HF_TOKEN` and other secrets.

Models are looked up **inside the repo first** (`models/mllm/`, `models/retriever/`), then fall back to paths such as `/workspace/models/`.

In [ ]:
from pathlib import Path

_cfgs = [REPO / "config/data.yaml", REPO / "config/model.yaml"]
_ok = 0
for p in _cfgs:
    ex = p.is_file()
    if ex:
        _ok += 1
    print(p.name, "found:" if ex else "missing:", p)
print("config files ready:", f"{_ok}/{len(_cfgs)}")


## Hugging Face model download

Weights are **not** included in the git repo. Use `util/download_models.py` to pull checkpoints listed in `config/model.yaml` into `models/` (Gemma, ColEmbed, optional fluency BART for metrics).

- Put **`HF_TOKEN`** or **`HUGGING_FACE_HUB_TOKEN`** in a repo-root **`.env`** if a model is gated (Gemma).
- **Layout after download:** `models/mllm/gemma-4-E4B-it/`, `models/retriever/llama-nemotron-colembed-vl-3b-v2/`, `models/eval/bart-large-cnn/`.
- **CLI:** `python util/download_models.py` (all), or `--only gemma colembed`, or `--dry-run` to print planned paths only.

For model names, paths, and Hugging Face setup, see the root **`README.md`**.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Requires REPO from "Resolve repo root" above; fallback for quick runs
try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "util" / "download_models.py").is_file() else _here.parent

dl = _r / "util" / "download_models.py"
if not dl.is_file():
    raise FileNotFoundError(f"Missing {dl}")

print("=== Download Hugging Face checkpoints (run) ===")
print("Script: util/download_models.py")
print("Targets: gemma, colembed (saved per config/model.yaml)")
print("Note: This can take a long time; gated models (e.g. Gemma) may need HF_TOKEN in .env.")
print("Logs from huggingface_hub / the script stream below.\n")

_dl_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
_rc_dl = subprocess.run(
    [sys.executable, "-u", str(dl), "--only", "gemma", "colembed"],
    cwd=str(_r),
    env=_dl_env,
    check=False,
)
print("\nDownload process exit code:", _rc_dl.returncode)


## Simple text QA (no RAG)

Load **Gemma 4 E4B IT** from `util/download_models.py` and generate answers **without retrieval, GraphRAG, or a slice**. This is independent of the pipeline (pattern/extraction/inference)—use it to sanity-check **load → question → answer**.

- First run may take time while weights load. With a GPU, the cell defaults to `bf16`.
- If the weights folder is missing, run the **Hugging Face model download** cell above first.


In [ ]:
# Prerequisite: run the "Resolve repo root and paths" cell so `REPO` exists.
import logging
import os
import sys
import time

if "REPO" not in globals():
    raise RuntimeError("Run the REPO/setup cell first.")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _setup_verbose_logging() -> None:
    fmt = logging.Formatter("%(levelname)s [%(name)s] %(message)s")
    root = logging.getLogger()
    if not any(
        isinstance(h, logging.StreamHandler) and getattr(h, "stream", None) is sys.stdout
        for h in root.handlers
    ):
        h = logging.StreamHandler(sys.stdout)
        h.setFormatter(fmt)
        root.addHandler(h)
    root.setLevel(logging.INFO)
    for name in ("mllm.hf_gemma_4_e4b_it", "transformers", "transformers.modeling_utils"):
        logging.getLogger(name).setLevel(logging.INFO)
    try:
        from transformers import logging as tf_logging

        tf_logging.set_verbosity_info()
    except Exception:
        pass


_setup_verbose_logging()

import torch
from util.llm_defaults import effective_gemma4_e4b_it_model_path
from mllm.hf_gemma_4_e4b_it import configured, generate_text

_gemma_dir = effective_gemma4_e4b_it_model_path()
print("Gemma weights path:", _gemma_dir)
print("torch / CUDA:", torch.__version__, "|", torch.cuda.is_available())
print("configured() (weights available):", configured())

if not configured():
    print("No weights directory. Run the 'Hugging Face model download' cell above, then try again.")
else:
    if torch.cuda.is_available():
        os.environ.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")
        print("dtype hint: GEMMA4_E4B_IT_TORCH_DTYPE =", os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE", "(default)"))

    question = "What is the capital of France? Answer in one short sentence."
    print("\n--- Question ---\n", question)
    print("\n--- Generating: first run may print model/processor load logs below, then generation ---\n")
    t0 = time.perf_counter()
    answer = generate_text(question, max_new_tokens=128)
    elapsed = time.perf_counter() - t0
    print("\n--- Answer ---\n", answer)
    print(f"\n--- Done (wall time {elapsed:.2f}s) ---")


## Toy dataset (shard 14)

If `data/webqa/webqa_shard14_toy/webqa_slice/` is already in the repository, **you do not need to rebuild it**.

Otherwise generate the toy slice (requires full WebQA + baseline files as in `scripts/build_shard14_toy_split.py` defaults):

```bash
python scripts/build_shard14_toy_split.py
```

PNG files may live under `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/` (or the paths recorded in `webqa_images.jsonl`). Run **Preview toy images** below to display a few samples in the notebook.

In [ ]:
TOY_SLICE = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

print("=== Toy dataset (WebQA slice) ===")
print("Dataset: WebQA — shard 14 **toy** slice (mini corpus; question / image / text JSONL)")
print("path:", TOY_SLICE.resolve())
print("Typical files: webqa_questions.jsonl (Q/A/meta), webqa_texts.jsonl (text chunks), webqa_images.jsonl (image index), webqa_tables.jsonl (optional)")

_names = (
    "webqa_questions.jsonl",
    "webqa_texts.jsonl",
    "webqa_images.jsonl",
    "webqa_tables.jsonl",
)
for _name in _names:
    _p = TOY_SLICE / _name
    if _p.is_file():
        _n = sum(1 for _ in _p.open(encoding="utf-8"))
        print(f"  {_name}: {_n} lines")
    else:
        print(f"  {_name}: (file missing)")

_qp = TOY_SLICE / "webqa_questions.jsonl"
if _qp.is_file():
    import json as _json
    from collections import Counter

    _qcates = Counter()
    _splits = Counter()
    _n_img_only = _n_txt_only = _n_both = _n_neither = 0
    _n_chars_q = 0
    _rows = 0
    for _line in _qp.open(encoding="utf-8"):
        _line = _line.strip()
        if not _line:
            continue
        _rows += 1
        _o = _json.loads(_line)
        _qcates[str(_o.get("Qcate") or "?")] += 1
        _splits[str(_o.get("split") or "?")] += 1
        _q = _o.get("Q") or ""
        _n_chars_q += len(_q)
        _hi = bool(_o.get("img_posFacts"))
        _ht = bool(_o.get("txt_posFacts"))
        if _hi and _ht:
            _n_both += 1
        elif _hi:
            _n_img_only += 1
        elif _ht:
            _n_txt_only += 1
        else:
            _n_neither += 1
    print("Question JSONL breakdown (webqa_questions.jsonl):")
    print("  rows:", _rows)
    if _rows:
        print("  avg question length (chars):", round(_n_chars_q / _rows, 1))
    print("  counts by Qcate:", dict(sorted(_qcates.items())))
    print("  counts by split:", dict(sorted(_splits.items())))
    print(
        "  positive-evidence modality — img+txt:",
        _n_both,
        "| img only:",
        _n_img_only,
        "| txt only:",
        _n_txt_only,
        "| neither:",
        _n_neither,
    )
else:
    print("webqa_questions.jsonl missing — cannot compute question-level stats.")


### Preview toy images (optional)

Reads `webqa_images.jsonl` and renders a handful of PNGs. When a path in the JSONL points somewhere like `/workspace/...` but the file is absent, we fall back to the same basename under `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/` inside the repository.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display

try:
    _slice = TOY_SLICE
except NameError:
    try:
        _slice = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"
    except NameError:
        _here = Path.cwd()
        _root = _here if (_here / "data" / "webqa").is_dir() else _here.parent
        _slice = _root / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "inference.py").is_file() else _here.parent

imgs_jsonl = _slice / "webqa_images.jsonl"
local_shard = _repo / "data" / "webqa" / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014"

if not imgs_jsonl.is_file():
    print("Missing:", imgs_jsonl)
else:
    rows = []
    with imgs_jsonl.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f"Images in slice index: {len(rows)}")
    for obj in rows[:5]:
        url = obj.get("url") or ""
        p = Path(url)
        if not p.is_file() and p.name:
            alt = local_shard / p.name
            if alt.is_file():
                p = alt
        if not p.is_file():
            print("Skip (file not found):", url[:80], "...")
            continue
        print(obj.get("image_id"), obj.get("title", "")[:60])
        display(IPyImage(filename=str(p), width=400))

## End-to-end run (recommended for learners)

`tests/test_pipeline.py` folds phases 0→6 together with the right environment variables, just like running from the shell.

**You may skip this section** if you are following **step-by-step (manual)** execution instead.

- **`-n 5`** — five questions (fast).
- **`--dry-run`** — no GPU LLM; placeholder answers (good for wiring checks).
- **`GEMMA4_E4B_IT_TORCH_DTYPE=bf16`** — recommended for real runs on limited VRAM.

In [ ]:
import os
import subprocess
import sys

# Default is real inference (GPU). True = dry-run (smoke / CPU friendly)
DRY_RUN = False  # default GPU + real weights (set True only for fast smoke tests)
N_QUERIES = 2

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
if not DRY_RUN:
    env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

cmd = [
    sys.executable,
    str(REPO / "tests" / "test_pipeline.py"),
    "-n", str(N_QUERIES),
]
if DRY_RUN:
    cmd.append("--dry-run")

print("End-to-end config  DRY_RUN=", DRY_RUN, " N_QUERIES=", N_QUERIES)
print("Command:", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(REPO), env=env)
print("exit code:", r.returncode)

# After the integrated test: point out latest result/ dirs and leaderboard metrics if present
print("\n--- End-to-end run output (hints) ---")
_rr = REPO / "result"
if _rr.is_dir():
    _runs = sorted(_rr.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for _p in _runs[:5]:
        _pred = _p / "phase5_inference" / "predictions.json"
        _rep = _p / "phase5_inference" / "evaluation_report.json"
        _flags = []
        if _pred.is_file():
            _flags.append("predictions")
        if _rep.is_file():
            _flags.append("evaluation_report")
        _tag = ", ".join(_flags) if _flags else "(no inference/eval artifacts)"
        print(f"  {_p.name}  |  {_tag}")
    if _runs:
        _latest_rep = _runs[0] / "phase5_inference" / "evaluation_report.json"
        if _latest_rep.is_file():
            import json as _json

            def _sf(_x):
                try:
                    return float(_x) if _x is not None else 0.0
                except (TypeError, ValueError):
                    return 0.0

            with _latest_rep.open(encoding="utf-8") as _f:
                _repd = _json.load(_f)
            _all_lb = (_repd.get("leaderboard_summary") or {}).get("All") or {}
            if _all_lb:
                print("\n--- Latest run: leaderboard (All) ---")
                print(
                    f"  QA-FL={_sf(_all_lb.get('QA-FL')):.4f}  "
                    f"QA-Acc={_sf(_all_lb.get('QA-Acc')):.4f}  "
                    f"QA={_sf(_all_lb.get('QA')):.4f}"
                )
                print("  report:", _latest_rep.resolve())
else:
    print("  No result/ directory yet — run the pipeline once.")


**After the run completes**, artifacts land under **`result/<RUN_ID>/`**:

- `webqa_slice/` — copied or exported JSONL
- `phase2_pattern_cache/`, `phase3_extraction_cache/`
- `phase4_graphs_out/*.graphml`
- `phase5_inference/predictions.json`, `evaluation_report.json`

In [ ]:
# Recent result runs
import datetime

result_root = REPO / "result"
if result_root.is_dir():
    runs = sorted(result_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in runs[:5]:
        ts = datetime.datetime.fromtimestamp(p.stat().st_mtime)
        pred = (p / "phase5_inference" / "predictions.json").is_file()
        rep = (p / "phase5_inference" / "evaluation_report.json").is_file()
        print(p.name, "| mtime:", ts.isoformat(sep=" ", timespec="seconds"))
        print("    predictions:", pred, "| evaluation_report:", rep)
else:
    print("No result/ yet — run the pipeline first.")


## Inspect one GraphML (Phase 4 output)

Graphs are standard NetworkX GraphML; we keep **nodes** / **edges** terminology below. Typical node attributes include `entity_name`, `type`, `description`, `source_id`.

**Cells in this section:** Load the first GraphML under `result/` via `nx.read_graphml`, print numeric stats plus a few sampled nodes, and (for small graphs) draw a spring-layout **matplotlib** preview. Huge graphs skip the plot.

**Interactive viz:** Under **Demo UI (optional)** you can spin up the Vite frontend; `GraphViewer` (`react-force-graph-2d`) renders question-specific graphs as a force-directed view from nodes/edges returned by the backend.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

graphs_dir = REPO / "result"
gml_files = list(graphs_dir.glob("**/phase4_graphs_out/*.graphml"))
if not gml_files:
    print("No GraphML under result/*/phase4_graphs_out/")
else:
    path = sorted(gml_files)[0]
    G = nx.read_graphml(path)
    print("file:", path.relative_to(REPO))
    print("nodes:", G.number_of_nodes(), "  edges:", G.number_of_edges())
    try:
        _Gu = G.to_undirected() if G.is_directed() else G
        print("density (undirected):", f"{nx.density(_Gu):.6f}")
    except Exception as _e:
        print("density (undirected) skipped:", _e)
    for nid in list(G.nodes)[:3]:
        print(" sample node:", nid, dict(G.nodes[nid]))

    _G = G.to_undirected() if G.is_directed() else G
    _n = _G.number_of_nodes()
    if _n == 0:
        pass
    elif _n > 300:
        print("plot skipped: too many nodes (", _n, ") — use Gephi/GraphML tooling for huge graphs")
    else:
        fig, ax = plt.subplots(figsize=(7, 5), dpi=110)
        k = 0.6 / max(1, _n ** 0.5)
        pos = nx.spring_layout(_G, seed=0, k=k, iterations=50)
        nx.draw_networkx(
            _G,
            pos=pos,
            ax=ax,
            with_labels=False,
            node_size=180,
            width=0.6,
            edge_color="gray",
            alpha=0.9,
        )
        ax.set_title("GraphML preview (spring layout) — first .graphml under result/")
        ax.axis("off")
        fig.tight_layout()
        plt.show()


## Step-by-step phases (manual)

Set `RUN_ID`, `N_QUERIES`, and `DRY_RUN` in the next cell and execute phases 0→6 in order. This mirrors `tests/test_pipeline.py` (toy data, local paths).

**You may skip or adjust this section** if you already ran **End-to-end** — at minimum use a **fresh `RUN_ID`** so outputs do not clash.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# --- knobs for your experiment ---
RUN_ID = "notebook_tutorial_run"  # change per experiment
N_QUERIES = 2
DRY_RUN = False  # default: GPU + real Gemma (set True only for smoke / CPU)

# --- paths (same logic as tests/test_pipeline.py) ---
_LOCAL_DATA = REPO / "data" / "webqa"
if (_LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice").is_dir():
    TOY_SLICE = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice"
else:
    TOY_SLICE = Path("/workspace/data/webqa/WebQA_imgs_7z_chunks/webqa_shard14_toy/webqa_slice")

if (_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014").is_dir():
    SHARD14_IMGS = str(_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014")
else:
    SHARD14_IMGS = "/workspace/data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014"

result_dir = REPO / "result" / RUN_ID
slice_dir = result_dir / "webqa_slice"
q_file = slice_dir / "webqa_questions.jsonl"
t_file = slice_dir / "webqa_texts.jsonl"

dry_run = "1" if DRY_RUN else "0"

from util.llm_defaults import DEFAULT_GEMMA4_E4B_IT_MODEL_PATH
gemma_path = os.environ.get("GEMMA4_E4B_IT_MODEL_PATH", "").strip() or str(DEFAULT_GEMMA4_E4B_IT_MODEL_PATH)

common_env = {
    "PYTHONPATH": str(REPO),
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "WEBQA_RUN_PROFILE": "val_n100",
    "WEBQA_DATA_ROOT": str(_LOCAL_DATA) if _LOCAL_DATA.is_dir() else "/workspace/data/webqa",
    "PATTERN_MAX_SAMPLES": str(N_QUERIES),
    "WEBQA_EXPORT_MAX": str(N_QUERIES),
    "EXTRACTION_MAX_QUESTIONS": str(N_QUERIES),
    "CONSTRUCT_MAX_QUESTIONS": str(N_QUERIES),
    "PATTERN_DRY_RUN": dry_run,
    "EXTRACTION_DRY_RUN": dry_run,
    "PATTERN_CONCURRENCY": "1",
    "EXTRACTION_CONCURRENCY": "1",
    "GEMMA4_E4B_IT_MODEL_PATH": gemma_path,
    "VIDORE_TEXT_LLM_BACKEND": "hf_gemma_4_e4b_it",
    "COLEMBED_MODEL_PATH": os.environ.get(
        "COLEMBED_MODEL_PATH",
        "/workspace/models/retriever/llama-nemotron-colembed-vl-3b-v2",
    ),
}
if os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE"):
    common_env["GEMMA4_E4B_IT_TORCH_DTYPE"] = os.environ["GEMMA4_E4B_IT_TORCH_DTYPE"]
elif not DRY_RUN:
    common_env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

def run_step(desc: str, cmd: list[str], env: dict) -> int:
    merged = {**os.environ, **env}
    print(f"\n=== {desc} ===\n{' '.join(cmd)}\n")
    r = subprocess.run(cmd, cwd=str(REPO), env=merged)
    print(f"exit code={r.returncode}")
    return r.returncode

print("RUN_ID:", RUN_ID, "| N_QUERIES:", N_QUERIES, "| DRY_RUN:", DRY_RUN)
print("Toy slice:", TOY_SLICE)

print("Result dir:", result_dir)
print("Slice dir (copied in the next cell):", slice_dir)
print("Image shard:", SHARD14_IMGS)
print("Gemma path:", gemma_path)
print("ColEmbed:", common_env.get("COLEMBED_MODEL_PATH"))


### Copy pre-built toy `webqa_slice` into `result/<RUN_ID>/`

In [ ]:
if not TOY_SLICE.is_dir():
    raise FileNotFoundError(f"Toy slice missing: {TOY_SLICE}")

result_dir.mkdir(parents=True, exist_ok=True)
if slice_dir.exists():
    shutil.rmtree(slice_dir)
shutil.copytree(TOY_SLICE, slice_dir)
print("Copy complete:", slice_dir)
_qpath = slice_dir / "webqa_questions.jsonl"
if _qpath.is_file():
    _nq = sum(1 for _ in _qpath.open(encoding="utf-8"))
    print("Questions in slice (webqa_questions.jsonl lines):", _nq)
else:
    print("(webqa_questions.jsonl missing)")


### Pattern (`pattern.py`)

**What we mean by “pattern”** matches [Query-Driven Multimodal GraphRAG (Bu et al., ACL 2025 Findings)](https://aclanthology.org/2025.findings-acl.1100/): **the scaffolding of a local knowledge graph anchored to query semantics.** The paper frames the approach as (1) deriving graph patterns from query semantics to guide extraction, (2) retrieving key knowledge via multi-path search, (3) adding multimodal context when helpful.

**Phase 2 in this notebook** asks an LLM, per question, for the structural **pattern** of the subgraph to build—for example which node/edge flavors matter first. Phase 3 **extraction** then fills entities and relations from text/image evidence guided by that pattern. In short: **pattern ≈ blueprint for question-specific graphs**, **extraction ≈ populating that blueprint.**



In [ ]:
pattern_cache = result_dir / "phase2_pattern_cache"
_local_pattern = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice" / "webqa_questions.jsonl"
pattern_question_file = str(_local_pattern) if _local_pattern.is_file() else "/workspace/data/webqa/WebQA_data_first_release/WebQA_train_val.json"

pattern_env = {
    **common_env,
    "PATTERN_JSON_FILE_PATH": pattern_question_file,
    "PATTERN_CACHE_DIR": str(pattern_cache),
}
_rc_p2 = run_step("Phase 2 pattern", [sys.executable, "pattern.py"], pattern_env)
print("Pattern cache:", pattern_cache, "| exists:", pattern_cache.is_dir())
if _rc_p2 != 0:
    print("Warning: phase 2 exited with code", _rc_p2)


### Extraction (`extraction.py`)

In [ ]:
extraction_cache = result_dir / "phase3_extraction_cache"
extraction_env = {
    **common_env,
    "EXTRACTION_QUESTION_FILE": str(q_file),
    "EXTRACTION_TEXT_FILE": str(t_file),
    "EXTRACTION_PATTERN_CACHE_DIR": str(pattern_cache),
    "EXTRACTION_CACHE_DIR": str(extraction_cache),
}
_rc_p3 = run_step("Phase 3 extraction", [sys.executable, "extraction.py"], extraction_env)
print("Extraction cache:", extraction_cache, "| exists:", extraction_cache.is_dir())
if _rc_p3 != 0:
    print("Warning: phase 3 exited with code", _rc_p3)


### Construct GraphML (`construct.py`)

In [ ]:
graph_dir = result_dir / "phase4_graphs_out"
construct_env = {
    **common_env,
    "CONSTRUCT_QUESTION_FILE": str(q_file),
    "CONSTRUCT_TABLE_FILE": str(slice_dir / "webqa_tables.jsonl"),
    "CONSTRUCT_IMAGE_FILE": str(slice_dir / "webqa_images.jsonl"),
    "CONSTRUCT_TEXT_FILE": str(t_file),
    "CONSTRUCT_EXTRACTION_CACHE": str(extraction_cache),
    "CONSTRUCT_OUTPUT_GRAPH_DIR": str(graph_dir),
    "CONSTRUCT_WEBQA_SLICE_DIR": str(slice_dir),
}
_rc_p4 = run_step("Phase 4 construct", [sys.executable, "construct.py"], construct_env)
_gml_list = list(graph_dir.glob("*.graphml")) if graph_dir.is_dir() else []
print("GraphML output dir:", graph_dir, "| file count:", len(_gml_list))
if _rc_p4 != 0:
    print("Warning: phase 4 exited with code", _rc_p4)


### Inference (`inference.py`)

When retrieval is enabled, ColEmbed participates; Gemma drafts answers (**skipped** when `DRY_RUN` is active).


In [ ]:
phase5_dir = result_dir / "phase5_inference"
phase5_dir.mkdir(parents=True, exist_ok=True)
pred_json = phase5_dir / "predictions.json"

inference_env = {
    **common_env,
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "INFERENCE_GRAPH_DIR": str(graph_dir),
    "INFERENCE_QUESTION_FILE": str(q_file),
    "INFERENCE_OUTPUT_JSON": str(pred_json),
    "INFERENCE_MAX_QUESTIONS": str(N_QUERIES),
    "INFERENCE_DRY_RUN": dry_run,
    "INFERENCE_COLEMBED_RETRIEVAL": os.environ.get("INFERENCE_COLEMBED_RETRIEVAL", "1"),
    "INFERENCE_WEBQA_SLICE_DIR": str(slice_dir),
    "WEBQA_IMGS_DIR": SHARD14_IMGS,
}
_rc_p5 = run_step("Phase 5 inference", [sys.executable, "inference.py"], inference_env)

# predictions.json is typically a mapping qid -> answer string
if pred_json.is_file():
    import json as _json

    with pred_json.open(encoding="utf-8") as _f:
        _preds = _json.load(_f)
    _keys = list(_preds.keys()) if isinstance(_preds, dict) else []
    print("\n--- Inference summary ---")
    print("Output file:", pred_json)
    print("prediction qids:", len(_keys))
    for _qid in _keys[:5]:
        _ans = str(_preds[_qid])[:240]
        print(f"  qid={_qid}  answer(prefix 240 chars): {_ans!r}")
    if len(_keys) > 5:
        print(f"  ... {len(_keys) - 5} more")
else:
    print("No inference output at:", pred_json)
if _rc_p5 != 0:
    print("Warning: phase 5 exited with code", _rc_p5)


### QA evaluation (`eval/evaluate_webqa_qa.py`)

In [ ]:
import json
from pathlib import Path


def _print_webqa_report_summary(report_path: Path) -> None:
    """Print leaderboard, raw scores, and retrieval summary from evaluation_report.json."""

    def _f(x) -> float:
        try:
            if x is None:
                return 0.0
            return float(x)
        except (TypeError, ValueError):
            return 0.0

    if not report_path.is_file():
        print("evaluation report missing:", report_path)
        return
    with report_path.open(encoding="utf-8") as f:
        rep = json.load(f)
    counts = rep.get("counts") or {}
    diag = rep.get("diagnostics") or {}

    print("\n" + "=" * 62)
    print("WebQA QA evaluation summary")
    print("=" * 62)
    print(
        "samples — all:",
        counts.get("All"),
        "| scored qids:",
        counts.get("scored"),
        "| unimodal:",
        counts.get("Unimodal"),
        "| multimodal:",
        counts.get("Multimodal"),
    )
    if diag.get("webqa_fluency_effective_backend"):
        print(
            "fluency backend:",
            diag.get("webqa_fluency_effective_backend"),
            "| model:",
            diag.get("webqa_fluency_model", ""),
        )

    lb = rep.get("leaderboard_summary") or {}
    labels = {"All": "all", "Unimodal": "unimodal", "Multimodal": "multimodal"}
    for bucket in ("All", "Unimodal", "Multimodal"):
        b = lb.get(bucket) or {}
        if not b:
            continue
        print(
            f"  [{labels[bucket]}]  QA-FL={_f(b.get('QA-FL')):.4f}  "
            f"QA-Acc={_f(b.get('QA-Acc')):.4f}  QA={_f(b.get('QA')):.4f}"
        )

    scores = rep.get("scores") or {}
    all_s = scores.get("All") or {}
    if all_s:
        print(
            "raw list F1 / list EM (all):",
            f"{_f(all_s.get('f1')):.4f}",
            "/",
            f"{_f(all_s.get('em')):.4f}",
        )
        print(
            "keyword list F1 / list EM (all):",
            f"{_f(all_s.get('list_f1_keyword')):.4f}",
            "/",
            f"{_f(all_s.get('list_em_keyword')):.4f}",
        )
        if all_s.get("webqa_acc_approx") is not None:
            print(f"approx WebQA acc (all): {_f(all_s.get('webqa_acc_approx')):.4f}")

    retr = rep.get("retrieval_summary") or {}
    if retr.get("status") == "missing_rankings_json":
        print("retrieval: retrieval_rankings.json missing (drop under result/<RUN_ID>/ when applicable)")
    elif retr and not retr.get("status"):
        k = int(retr.get("k_gen", 10))
        overall = retr.get("overall") or {}
        metrics_block = overall.get("metrics") or {}
        m = metrics_block.get(f"k={k}")
        if m:
            print(
                f"retrieval@{k} (overall)  hit@k={_f(m.get('hit@k')):.4f}  "
                f"recall@k={_f(m.get('recall@k')):.4f}  "
                f"MRR={_f(m.get('mrr@k')):.4f}  "
                f"retrieval F1={_f(m.get('retrieval_f1')):.4f}"
            )

    by_acc = scores.get("by_Qcate_webqa_acc_approx") or {}
    if by_acc:
        parts = [f"{c}={_f(v):.3f}" for c, v in sorted(by_acc.items())]
        print("approx WebQA acc by Qcate:", " | ".join(parts))
    by_qa = scores.get("by_Qcate_webqa_qa") or {}
    if by_qa:
        parts = [f"{c}={_f(v):.3f}" for c, v in sorted(by_qa.items())]
        print("QA (leaderboard) by Qcate:", " | ".join(parts))

    print("full JSON:", report_path.resolve())
    print("=" * 62 + "\n")


if pred_json.is_file():
    report_json = phase5_dir / "evaluation_report.json"
    eval_env = {**common_env, "MMGRAPHRAG_RUN_ID": RUN_ID}
    _rc_p6 = run_step(
        "Phase 6 eval",
        [
            sys.executable,
            "eval/evaluate_webqa_qa.py",
            "--predictions",
            str(pred_json),
            "--gold_jsonl",
            str(q_file),
            "--report_json",
            str(report_json),
            "--split_label",
            "val",
        ],
        eval_env,
    )
    if _rc_p6 == 0:
        _print_webqa_report_summary(report_json)
    else:
        print("eval script exited with errors — inspect logs above. exit code:", _rc_p6)
else:
    print("Skipping eval: predictions.json missing")


## Appendix: raw shell commands (optional)

The recommended path is **Step-by-step (manual)** in this notebook; below we list **bash** snippets for terminals-only setups. Customize `$RUN_ID`, paths, and limits for your machine.

**Phase 0 — export slice** (skip if you already copied `webqa_slice` into `result/$RUN_ID/`):
```bash
export MMGRAPHRAG_RUN_ID=my_run WEBQA_EXPORT_MAX=5 WEBQA_SLICE_DIR=result/my_run/webqa_slice
python export_webqa_slice.py
```

**Phase 2 — pattern:**
```bash
export PATTERN_MAX_SAMPLES=5 PATTERN_CACHE_DIR=result/my_run/phase2_pattern_cache
python pattern.py
```

**Phase 3 — extraction:**
```bash
export EXTRACTION_QUESTION_FILE=result/my_run/webqa_slice/webqa_questions.jsonl ...
python extraction.py
```

Environment-variable wiring mirrors `tests/test_pipeline.py`; copy entries from there when you need a reproducible recipe.


## Demo UI (optional)

After you have a `result/<RUN_ID>/` with predictions and graphs, configure `demo/be/config/paths.yaml` (often `run_id: latest`). See `demo/README.md`.

Either run the **terminal commands** above or kick off backends with the **Start demo BE + FE** cell (`subprocess`). Requirements:

- **BE:** same Python venv as the pipeline; ports **8000** (override with env `DEMO_BE_PORT`).
- **FE:** **Node.js + npm** (e.g. LTS via nvm); port **5173** (`DEMO_FE_PORT`). Vite proxies `/api` to the backend.

Open in the browser: **http://127.0.0.1:5173/** (FE) — API health: **http://127.0.0.1:8000/health**.

### Run the same commands in a terminal

```bash
cd demo/be && python server.py --host 0.0.0.0 --port 8000
cd demo/fe && npm install && npm run dev
```

### Start demo BE + FE from the notebook

Servers run in the **background** so you can keep working in Jupyter. Execute **Resolve repo root and paths** first so `REPO` exists (otherwise the cell falls back to the current directory).

Re-running this notebook cell stops any demo processes registered in `DEMO_PROCESSES` for this kernel session.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "demo" / "be" / "server.py").is_file() else _here.parent

be_dir = _repo / "demo" / "be"
fe_dir = _repo / "demo" / "fe"
if not (be_dir / "server.py").is_file():
    raise FileNotFoundError(f"Expected demo backend at {be_dir}")

if "DEMO_PROCESSES" not in globals():
    DEMO_PROCESSES = []
else:
    for _p in list(DEMO_PROCESSES):
        if _p.poll() is None:
            _p.terminate()
    DEMO_PROCESSES.clear()

BE_PORT = int(os.environ.get("DEMO_BE_PORT", "8000"))
FE_PORT = int(os.environ.get("DEMO_FE_PORT", "5173"))

be_env = {**os.environ, "PYTHONPATH": str(be_dir)}
p_be = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", str(BE_PORT)],
    cwd=str(be_dir),
    env=be_env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
DEMO_PROCESSES.append(p_be)
print(f"Demo BE started | pid={p_be.pid} | http://127.0.0.1:{BE_PORT}/health")
time.sleep(1.5)

npm = shutil.which("npm")
if npm is None:
    print("npm not in PATH — install Node.js, then: cd demo/fe && npm install && npm run dev")
else:
    if not (fe_dir / "node_modules").is_dir():
        print("Running npm install in demo/fe (first time may take a minute)...")
        subprocess.run([npm, "install"], cwd=str(fe_dir), check=False)
    p_fe = subprocess.Popen(
        [npm, "run", "dev"],
        cwd=str(fe_dir),
        env={**os.environ},
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    DEMO_PROCESSES.append(p_fe)
    print(f"Demo FE started | pid={p_fe.pid} | http://127.0.0.1:{FE_PORT}/")

print("\nOpen the frontend URL above. Stop servers with the next **Stop demo servers** cell.")

### Stop demo servers

Stops tracked processes from **Start demo BE + FE** (`DEMO_PROCESSES`). If ports **8000** / **5173** remain busy, terminate them manually (`kill`, Task Manager, etc.).


In [ ]:
import time as _time

if "DEMO_PROCESSES" in globals() and DEMO_PROCESSES:
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.terminate()
    _time.sleep(0.5)
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.kill()
    DEMO_PROCESSES.clear()
    print("Demo BE/FE processes terminated.")
else:
    print("No DEMO_PROCESSES to stop (run Start demo BE + FE first).")

## Further reading

- `README.md` — overview, data layout, model download, and pipeline overview.
- `tests/test_pipeline.py` — full orchestration and environment variables for each phase.